# Synthesis with fundamental frequency and inharmonicity

This notebook **only synthesises sound**. There are no target/initial pairs,
losses, optimisation controls or backward passes.

Run all cells once, edit the controls, and click **Synthesise**. Use the project's
Pixi kernel (`pixi install` after updating). The first synthesis may compile the
native C++ kernel. The mathematical mapping and synthesis helpers are defined
below so you can inspect them.

## 1. What do the two controls mean?

For the small-amplitude, undamped, simply supported stiff string,

\[
f_n=\frac{n}{2L}\sqrt{\frac{T}{\mu}}\sqrt{1+Bn^2},\qquad
\mu=\rho A,\quad A=\pi r^2,\quad I=\frac{\pi r^4}{4},\quad
B=\frac{\pi^2 EI}{TL^2}.
\]

Here **(f_1)** means the first mode's frequency **including stiffness**. It is
not the tension-only frequency \(\sqrt{T/\mu}/(2L)\). Expressing every partial
relative to this fundamental gives

\[
\boxed{f_n=n f_1\sqrt{\frac{1+Bn^2}{1+B}}}.
\]

- (B=0): exact harmonic spacing in the continuous linear model.
- Larger (B): higher modes become progressively sharper relative to (n f_1).
- Changing (B) at fixed (f_1) keeps the **linear fundamental** fixed while
  changing partial spacing.

There are many physical strings with the same (f_1,B). We choose a definite
mapping by setting density \(\rho\), radius \(r\), and length \(L\), then deriving
**both** tension and Young's modulus:

\[
\boxed{T=\frac{4\mu L^2 f_1^2}{1+B}},\qquad
\boxed{E=\frac{B T L^2}{\pi^2 I}}.
\]

Thus changing (B) changes both (T) and (E); it does not just turn a stiffness
knob while leaving tension fixed. At (B=0), this mapping gives (E=0), so it also
removes the KC stretching nonlinearity. This is a mathematical synthesis limit,
not a realistic elastic material.

The rendered fundamental can differ slightly from (f_1): the finite grid has
numerical dispersion, damping affects oscillation, and finite excitation causes
nonlinear pitch shifts. Use a small amplitude to compare against the linear
partial formula.


In [ ]:
%matplotlib inline
from pathlib import Path
import sys
import math
import time
import torch
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import Audio, HTML, display, clear_output

root = next(path for path in [Path.cwd(), *Path.cwd().parents]
            if (path / "src" / "kc_string.py").exists())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from src import kc_string as reference
from src.kc_string_forward import native_block

torch.set_num_threads(1)


In [ ]:
def physical_parameters(fundamental_hz, inharmonicity, density, radius, length):
    """Map the continuous linear fundamental f1 and B to physical T and E."""
    values = [fundamental_hz, inharmonicity, density, radius, length]
    if not all(math.isfinite(value) for value in values):
        raise ValueError("All parameters must be finite.")
    if min(fundamental_hz, density, radius, length) <= 0 or inharmonicity < 0:
        raise ValueError("Use f1, density, radius and length > 0, and B >= 0.")
    area = math.pi * radius**2
    inertia = math.pi * radius**4 / 4
    mass_per_length = density * area
    tension = 4 * mass_per_length * length**2 * fundamental_hz**2 / (1 + inharmonicity)
    young_modulus = inharmonicity * tension * length**2 / (math.pi**2 * inertia)
    return dict(density=density, radius=radius, length=length, area=area,
                inertia=inertia, mass_per_length=mass_per_length,
                tension=tension, young_modulus=young_modulus)


def partial_frequencies(fundamental_hz, inharmonicity, count=8):
    mode = torch.arange(1, count + 1, dtype=torch.float64)
    return mode * fundamental_hz * torch.sqrt((1 + inharmonicity * mode**2) / (1 + inharmonicity))


def synthesise(fundamental_hz, inharmonicity, *, density=reference.material_density,
               radius=reference.radius, length=reference.length,
               amplitude=reference.amplitude, sigma_0=reference.sigma_0,
               sigma_1=reference.sigma_1, sample_rate=44100, duration=4.,
               oversampling=1, max_intervals=128, excitation="multimode"):
    """Value-only synthesis; the native kernel's unused derivative columns are zero.

    Same non-SAV update and startup convention as the reference. The multimode
    option changes only the initial displacement shape. No gradients or losses
    are requested, and there is no optimisation.
    """
    physical = physical_parameters(fundamental_hz, inharmonicity, density, radius, length)
    if not all(math.isfinite(v) for v in (amplitude, sigma_0, sigma_1, duration)):
        raise ValueError("Amplitude, damping and duration must be finite.")
    if amplitude <= 0 or min(sigma_0, sigma_1) < 0 or duration <= 0:
        raise ValueError("Use positive amplitude/duration and nonnegative damping.")
    if sample_rate <= 0 or oversampling < 1 or max_intervals < 4 or max_intervals % 2:
        raise ValueError("Use a positive rate, positive integer oversampling and an even grid limit >= 4.")
    if excitation not in ("first_mode", "multimode"):
        raise ValueError("Unknown excitation shape.")
    integration_rate = sample_rate * oversampling
    time_step = 1 / integration_rate
    mass = physical["mass_per_length"]
    tension = physical["tension"]
    modulus = physical["young_modulus"]
    inertia = physical["inertia"]
    c_squared = tension / mass
    K_squared = modulus * inertia / mass
    minimum_spacing = reference.minimum_grid_spacing(time_step, math.sqrt(c_squared), math.sqrt(K_squared), sigma_1) * 1.01
    maximum_stable_intervals = 2 * math.floor(length / minimum_spacing / 2)
    intervals = min(max_intervals, maximum_stable_intervals)
    if intervals < 4:
        raise ValueError("Too few stable spatial intervals. Increase sample rate or oversampling.")
    spacing = length / intervals
    steps = round(duration * integration_rate)
    if steps > 10_000_000:
        raise ValueError("This interactive notebook limits a synthesis to 10 million integration steps; reduce duration or oversampling.")
    points = torch.arange(1, intervals, dtype=torch.float64)
    modes = 1 if excitation == "first_mode" else min(8, intervals - 1)
    shape = sum(torch.sin(mode * math.pi * points / intervals) / mode**2 for mode in range(1, modes + 1))
    previous = amplitude * shape / shape.abs().max()
    second = reference.second_spatial_derivative(previous, spacing)
    fourth = reference.second_spatial_derivative(second, spacing)
    nonlinear = -torch.dot(previous, second) * second
    current = previous + .5*time_step**2 * (
        spacing*c_squared*second - spacing*K_squared*fourth
        + spacing*(.5*modulus*spacing/(length*density))*nonlinear)
    inverse_mass = 1 / (mass + mass*sigma_0*time_step)
    q_scale = .5*time_step*math.sqrt(spacing*modulus*physical["area"]/length)
    # Column zero contains values. Other columns are zero: no parameter tangents.
    states = torch.zeros((2, intervals-1, 9), dtype=torch.float64)
    states[0, :, 0], states[1, :, 0] = previous, current
    coefficients = torch.zeros((8, 9), dtype=torch.float64)
    coefficients[:, 0] = torch.tensor([
        1/spacing**2, mass, tension*time_step**2 + 2*mass*sigma_1*time_step,
        modulus*inertia*time_step**2, mass*sigma_0*time_step-mass,
        2*mass*sigma_1*time_step, inverse_mass, q_scale], dtype=torch.float64)
    with torch.no_grad():
        _, _, following = native_block(states, coefficients, torch.zeros(9, dtype=torch.float64), steps)
        midpoint = (intervals-1)//2
        audio = torch.cat((torch.stack((previous[midpoint], current[midpoint])), following))
        if oversampling > 1:
            half_width = 16*oversampling
            index = torch.arange(-half_width, half_width+1, dtype=torch.float64)
            cutoff = .45/oversampling
            kernel = 2*cutoff*torch.sinc(2*cutoff*index)*torch.hann_window(index.numel(), periodic=False, dtype=torch.float64)
            kernel = (kernel/kernel.sum()).reshape(1, 1, -1)
            padded = torch.nn.functional.pad(audio.reshape(1,1,-1), (half_width,half_width), mode="replicate")
            audio = torch.nn.functional.conv1d(padded, kernel, stride=oversampling).flatten()
        audio = audio[:round(duration*sample_rate)]
    if not torch.isfinite(audio).all():
        raise ValueError("Nonfinite simulation: reduce amplitude or choose safer settings.")
    mode = torch.arange(1, min(8, intervals-1)+1, dtype=torch.float64)
    eigenvalue = (2/spacing*torch.sin(mode*math.pi/(2*intervals)))**2
    discrete_linear_hz = integration_rate/math.pi * torch.asin(.5*time_step*torch.sqrt(c_squared*eigenvalue+K_squared*eigenvalue**2))
    return dict(audio=audio, sample_rate=sample_rate, parameters=physical,
                intervals=intervals, maximum_stable_intervals=maximum_stable_intervals,
                integration_rate=integration_rate, excited_modes=modes,
                fundamental_hz=fundamental_hz, inharmonicity=inharmonicity,
                linear_partials_hz=partial_frequencies(fundamental_hz,inharmonicity,len(mode)),
                discrete_linear_partials_hz=discrete_linear_hz)


## 2. Set the synthesis controls

The defaults reproduce the reference's **physical parameters**, expressed as
(f_1) and (B). The default **multimode** excitation adds the first eight spatial
sine modes with weights (1/n^2), scaled to the chosen peak displacement, so the
higher partials can be heard. Select **First mode (reference)** for the original
excitation instead. The midpoint listener suppresses even spatial modes.

Density, radius and length stay at the reference values unless you change the
advanced controls. They define which physical string corresponds to (f_1,B).
Damping coefficients are independent controls. Large excitation can shift pitch.

This synthesis notebook chooses an even grid at or below the requested maximum
to satisfy the **linear** stability bound with the reference's 1% spacing margin.
It reports the chosen grid; changing controls can change that grid. This is not
a full nonlinear stability proof. Oversampling is fixed for each synthesis and
uses a windowed-sinc low-pass filter before decimation.


In [ ]:
reference_B = math.pi**2 * reference.young_modulus * reference.moment_of_inertia / (reference.tension * reference.length**2)
reference_f1 = reference.wave_speed / (2*reference.length) * math.sqrt(1+reference_B)

controls = {
    "fundamental_hz": widgets.FloatText(value=reference_f1, description="Fundamental f₁ (Hz)"),
    "inharmonicity": widgets.FloatText(value=reference_B, description="Inharmonicity B"),
    "sample_rate": widgets.BoundedIntText(value=44100,min=1000,max=384000,description="Audio rate (Hz)"),
    "duration": widgets.BoundedFloatText(value=4.,min=.001,max=60.,description="Duration (s)"),
    "oversampling": widgets.BoundedIntText(value=1,min=1,max=16,description="Oversampling"),
    "max_intervals": widgets.BoundedIntText(value=128,min=4,max=4096,description="Max grid intervals"),
    "amplitude": widgets.FloatText(value=reference.amplitude,description="Amplitude (m)"),
    "sigma_0": widgets.FloatText(value=reference.sigma_0,description="Damping σ₀ (1/s)"),
    "sigma_1": widgets.FloatText(value=reference.sigma_1,description="Damping σ₁ (m²/s)"),
    "excitation": widgets.Dropdown(options=[("Several modes (hear partials)","multimode"),("First mode (reference)","first_mode")],value="multimode",description="Excitation"),
    "density": widgets.FloatText(value=reference.material_density,description="Density (kg/m³)"),
    "radius": widgets.FloatText(value=reference.radius,description="Radius (m)"),
    "length": widgets.FloatText(value=reference.length,description="Length (m)"),
}
for control in controls.values():
    control.style.description_width="170px"
    control.layout.width="420px"
advanced=widgets.Accordion(children=[widgets.VBox([controls[k] for k in ("density","radius","length")])],selected_index=None)
advanced.set_title(0,"Advanced: density and geometry")
display(widgets.VBox([controls[k] for k in controls if k not in ("density","radius","length")]+[advanced]))


## 3. Synthesise, inspect and play

Click **Synthesise** after changing settings. Playback is normalised for listening;
the stored displacement signal is not changed. The spectrum's dashed lines mark
the **continuous linear** partial predictions. Their locations are guides, not
promises of the exact nonlinear, damped discrete-model peaks.

`last_synthesis` stores the audio, sampling rate, derived physical parameters,
chosen grid and predicted partial frequencies. No optimisation takes place.


In [ ]:
synthesis_output = widgets.Output()
synthesise_button = widgets.Button(description="Synthesise",button_style="primary")
last_synthesis = None

def render_synthesis(_=None):
    global last_synthesis
    synthesise_button.disabled=True
    with synthesis_output:
        clear_output(wait=True)
        try:
            settings={name:control.value for name,control in controls.items()}
            started=time.perf_counter()
            result=synthesise(**settings)
            last_synthesis=result
            physical=result["parameters"]
            print(f"Derived tension T = {physical['tension']:.8g} N; Young's modulus E = {physical['young_modulus']:.8g} Pa")
            print(f"Grid: {result['intervals']} intervals; integration rate: {result['integration_rate']:,} Hz; synthesis: {time.perf_counter()-started:.2f} s")
            print(f"Continuous linear f₁: {result['fundamental_hz']:.6g} Hz; undamped discrete prediction: {result['discrete_linear_partials_hz'][0]:.6g} Hz")
            audio=result['audio']
            display(Audio(audio.numpy(),rate=result['sample_rate']))
            figure,axes=plt.subplots(1,2,figsize=(13,3.5),layout='constrained')
            times=torch.arange(audio.numel(),dtype=torch.float64)/result['sample_rate']
            axes[0].plot(times.numpy(),audio.numpy(),linewidth=.6)
            axes[0].set(xlabel='Time (s)',ylabel='Displacement (m)',title='Synthesised waveform')
            spectrum=torch.fft.rfft(audio*torch.hann_window(audio.numel(),dtype=torch.float64))
            frequency=torch.fft.rfftfreq(audio.numel(),1/result['sample_rate'])
            db=20*torch.log10((spectrum.abs()/spectrum.abs().max().clamp_min(1e-30)).clamp_min(1e-6))
            axes[1].plot(frequency.numpy(),db.numpy(),linewidth=.8)
            visible=result['linear_partials_hz']
            limit=min(result['sample_rate']/2, max(1000.,visible[-1].item()*1.15))
            for mode,partial in enumerate(visible.tolist(),1):
                if partial<limit:
                    axes[1].axvline(partial,color='orange',alpha=.45,linestyle='--')
                    axes[1].text(partial,-8,str(mode),fontsize=8,ha='center')
            axes[1].set(xlim=(0,limit),ylim=(-100,3),xlabel='Frequency (Hz)',ylabel='Magnitude (dB relative to peak)',title='Spectrum and linear mode predictions')
            display(figure);plt.close(figure)
            rows=''.join(f'<tr><td>{i}</td><td>{f:.5f}</td><td>{d:.5f}</td></tr>' for i,(f,d) in enumerate(zip(result['linear_partials_hz'].tolist(),result['discrete_linear_partials_hz'].tolist()),1))
            display(HTML('<table><tr><th>Mode</th><th>Continuous linear Hz</th><th>Undamped discrete Hz</th></tr>'+rows+'</table>'))
        except Exception as error:
            last_synthesis=None
            print(f'Synthesis not generated: {error}')
        finally:
            synthesise_button.disabled=False

def controls_changed(_):
    global last_synthesis
    last_synthesis=None
    with synthesis_output:
        clear_output(wait=False)
        print('Controls changed. Click Synthesise to generate a new sound.')

for control in controls.values(): control.observe(controls_changed,names='value')
synthesise_button.on_click(render_synthesis)
display(widgets.VBox([synthesise_button,synthesis_output]))
